## 1. 파일 읽기

In [1]:
import glob
import pandas as pd

# 배민 파일 31개를 하나의 DataFrame으로 합치기
baemin_files = sorted(glob.glob("data/baemin/baemin*.csv"))

baemin_list = []

for file in baemin_files:
    df_temp = pd.read_csv(file)
    baemin_list.append(df_temp)

baemin_df = pd.concat(baemin_list, ignore_index=True)


# 쿠팡 파일 31개를 하나의 DataFrame으로 합치기
coupang_files = sorted(glob.glob("data/coupang/coupang*.csv"))

coupang_list = []

for file in coupang_files:
    df_temp = pd.read_csv(file)
    coupang_list.append(df_temp)

coupang_df = pd.concat(coupang_list, ignore_index=True)


# 결과 확인
print("===== 배민 DataFrame =====")
print(baemin_df)

print("===== 쿠팡 DataFrame =====")
print(coupang_df)

===== 배민 DataFrame =====
                 order_at  total_bill ad_yn  cpc
0     2026-10-01 11:48:34       26000     Y  700
1     2026-10-01 11:53:22       15000     N    0
2     2026-10-01 12:18:52       35000     N    0
3     2026-10-01 12:28:44       35000     N    0
4     2026-10-01 12:50:50       35000     N    0
...                   ...         ...   ...  ...
1150  2026-10-31 20:34:53       17000     N    0
1151  2026-10-31 20:47:07       38000     Y  640
1152  2026-10-31 21:39:35       19000     Y  690
1153  2026-10-31 21:45:29       45000     N    0
1154  2026-10-31 21:51:03       19000     N    0

[1155 rows x 4 columns]
===== 쿠팡 DataFrame =====
              order_time  total_price advertisement_yn  ad_pct
0    2026-10-01 11:46:07        18000                Y      10
1    2026-10-01 12:06:50        21000                Y      10
2    2026-10-01 12:16:29        58000                Y      10
3    2026-10-01 12:19:49        43000                Y      10
4    2026-10-01 13:00:

## 2. 칼럼명 통일

In [2]:
# 두 플랫폼의 데이터프레임 칼럼명 변경
baemin_df = baemin_df.rename(
    columns={
        "order_at": "주문일시",
        "total_bill": "전체금액",
        "ad_yn": "광고유입여부",
    }
)

coupang_df = coupang_df.rename(
    columns={
        "order_time": "주문일시",
        "total_price": "전체금액",
        "advertisement_yn": "광고유입여부",
    }
)

# 플랫폼 칼럼 추가
baemin_df["플랫폼"] = "배민"
coupang_df["플랫폼"] = "쿠팡"

# 광고집행비 칼럼 추가
baemin_df["광고집행비"] = baemin_df["cpc"]
coupang_df["광고집행비"] = coupang_df["전체금액"] * coupang_df["ad_pct"] / 100

# 결과 확인
print("===== 배민 DataFrame =====")
print(baemin_df)

print("===== 쿠팡 DataFrame =====")
print(coupang_df)

===== 배민 DataFrame =====
                     주문일시   전체금액 광고유입여부  cpc 플랫폼  광고집행비
0     2026-10-01 11:48:34  26000      Y  700  배민    700
1     2026-10-01 11:53:22  15000      N    0  배민      0
2     2026-10-01 12:18:52  35000      N    0  배민      0
3     2026-10-01 12:28:44  35000      N    0  배민      0
4     2026-10-01 12:50:50  35000      N    0  배민      0
...                   ...    ...    ...  ...  ..    ...
1150  2026-10-31 20:34:53  17000      N    0  배민      0
1151  2026-10-31 20:47:07  38000      Y  640  배민    640
1152  2026-10-31 21:39:35  19000      Y  690  배민    690
1153  2026-10-31 21:45:29  45000      N    0  배민      0
1154  2026-10-31 21:51:03  19000      N    0  배민      0

[1155 rows x 6 columns]
===== 쿠팡 DataFrame =====
                    주문일시   전체금액 광고유입여부  ad_pct 플랫폼   광고집행비
0    2026-10-01 11:46:07  18000      Y      10  쿠팡  1800.0
1    2026-10-01 12:06:50  21000      Y      10  쿠팡  2100.0
2    2026-10-01 12:16:29  58000      Y      10  쿠팡  5800.0
3    2026-10-01 1

## 3. 데이터 통합

In [3]:
# 두 DataFrame 하나로 합치기
total_df = pd.concat([baemin_df, coupang_df], ignore_index=True)

# '주문일시'를 datetime 형식으로 변환하고 날짜만 담은 '주문일자' 칼럼 추가
total_df["주문일시"] = pd.to_datetime(total_df["주문일시"])
total_df["주문일자"] = total_df["주문일시"].dt.date

# 주문일시 순으로 정렬
total_df = total_df.sort_values(by="주문일시").reset_index(drop=True)

# 결과 확인
print("===== 배달어플 통합 Dataframe =====")
print(total_df)

===== 배달어플 통합 Dataframe =====
                    주문일시   전체금액 광고유입여부    cpc 플랫폼   광고집행비  ad_pct        주문일자
0    2026-10-01 11:46:07  18000      Y    NaN  쿠팡  1800.0    10.0  2026-10-01
1    2026-10-01 11:48:34  26000      Y  700.0  배민   700.0     NaN  2026-10-01
2    2026-10-01 11:53:22  15000      N    0.0  배민     0.0     NaN  2026-10-01
3    2026-10-01 12:06:50  21000      Y    NaN  쿠팡  2100.0    10.0  2026-10-01
4    2026-10-01 12:16:29  58000      Y    NaN  쿠팡  5800.0    10.0  2026-10-01
...                  ...    ...    ...    ...  ..     ...     ...         ...
1862 2026-10-31 20:47:07  38000      Y  640.0  배민   640.0     NaN  2026-10-31
1863 2026-10-31 20:49:23  24000      N    NaN  쿠팡     0.0     0.0  2026-10-31
1864 2026-10-31 21:39:35  19000      Y  690.0  배민   690.0     NaN  2026-10-31
1865 2026-10-31 21:45:29  45000      N    0.0  배민     0.0     NaN  2026-10-31
1866 2026-10-31 21:51:03  19000      N    0.0  배민     0.0     NaN  2026-10-31

[1867 rows x 8 columns]


## 4. 일자별 매출 집계

In [4]:
# 행(주문일자), 열(플랫폼), 값(전체금액 합계)으로 표 생성
pivot_df = total_df.pivot_table(
    index="주문일자", 
    columns="플랫폼", 
    values="전체금액", 
    aggfunc="sum",
)

# 합계열 추가
pivot_df["합계"] = pivot_df["배민"] + pivot_df["쿠팡"]

# 결과 확인
print("===== 일자별 매출 집계 =====")
print(pivot_df)

===== 일자별 매출 집계 =====
플랫폼              배민       쿠팡       합계
주문일자                                 
2026-10-01   959000   562000  1521000
2026-10-02  1348000   904000  2252000
2026-10-03  1294000   779000  2073000
2026-10-04  1127000   928000  2055000
2026-10-05  1103000   543000  1646000
2026-10-06  1028000   576000  1604000
2026-10-07  1227000   549000  1776000
2026-10-08   946000   439000  1385000
2026-10-09  1200000   835000  2035000
2026-10-10  1379000   719000  2098000
2026-10-11  1519000   979000  2498000
2026-10-12   838000   622000  1460000
2026-10-13   971000   539000  1510000
2026-10-14   981000   663000  1644000
2026-10-15  1068000   546000  1614000
2026-10-16  1420000   776000  2196000
2026-10-17  1382000  1038000  2420000
2026-10-18  1352000  1006000  2358000
2026-10-19   913000   625000  1538000
2026-10-20   815000   611000  1426000
2026-10-21   907000   435000  1342000
2026-10-22   946000   630000  1576000
2026-10-23  1309000   792000  2101000
2026-10-24  1295000  1007000

## 5. 광고 효과 분석

In [5]:
# 플랫폼별 광고유입 주문의 비율
mask = total_df["광고유입여부"] == "Y"
ad_counts = total_df[mask].groupby("플랫폼")["광고유입여부"].count()

total_counts = total_df.groupby("플랫폼")["광고유입여부"].count()

ad_ratio = round(ad_counts / total_counts * 100, 2)

# 플랫폼별 광고집행비 합계
ad_sum = total_df.groupby("플랫폼")["광고집행비"].sum()

#결과 확인
print("===== 플랫폼별 광고 유입 주문 비율(%) =====")
print(ad_ratio)

print("===== 플랫폼별 광고집행비 합계 =====")
print(ad_sum)

===== 플랫폼별 광고 유입 주문 비율(%) =====
플랫폼
배민    34.98
쿠팡    51.97
Name: 광고유입여부, dtype: float64
===== 플랫폼별 광고집행비 합계 =====
플랫폼
배민    243170.0
쿠팡    886700.0
Name: 광고집행비, dtype: float64


## 6. 식자재비 추출

In [6]:
# 은행 거래내역에서 거래처가 식자재마트인 출금 내역
bank_df = pd.read_csv("data/bank_20261001_20261031.csv", encoding="euc-kr")

mart_df = bank_df[bank_df["거래처"] == "식자재마트"]

print("===== 식자재마트 출금내역 =====")
print(mart_df)

===== 식자재마트 출금내역 =====
                 거래일시    적요    거래처     출금액  입금액        잔액
1    2026.10.01 09:01  체크카드  식자재마트  197100    0   4796800
5    2026.10.02 09:15  체크카드  식자재마트  267100    0   6193900
7    2026.10.02 15:03  체크카드  식자재마트  113300    0   6056000
10   2026.10.03 09:07  체크카드  식자재마트  194800    0   5437000
11   2026.10.03 15:28  체크카드  식자재마트   52800    0   5384200
13   2026.10.04 09:08  체크카드  식자재마트  289300    0   5082700
15   2026.10.05 09:44  체크카드  식자재마트  214400    0   4859900
20   2026.10.06 09:04  체크카드  식자재마트  209500    0   6969700
24   2026.10.07 09:03  체크카드  식자재마트  273100    0   6540300
26   2026.10.07 15:16  체크카드  식자재마트   59700    0   6470700
28   2026.10.08 09:02  체크카드  식자재마트  219300    0   6245800
32   2026.10.09 09:28  체크카드  식자재마트  257300    0   7656800
34   2026.10.09 15:46  체크카드  식자재마트   94000    0   7542400
37   2026.10.10 09:42  체크카드  식자재마트  250700    0   6867700
38   2026.10.10 15:42  체크카드  식자재마트   74200    0   6793500
41   2026.10.11 09:39  체크카드  식자재마트  225000    0  

In [7]:
# 일자별 식자재비 합계
mart_df["거래일자"] = pd.to_datetime(mart_df["거래일시"]).dt.date
daily_mart_cost = mart_df.groupby("거래일자")["출금액"].sum()

print("===== 일자별 식자재비 합계 ======")
print(daily_mart_cost)

===== 일자별 식자재비 합계 ======
거래일자
2026-10-01    197100
2026-10-02    380400
2026-10-03    247600
2026-10-04    289300
2026-10-05    214400
2026-10-06    209500
2026-10-07    332800
2026-10-08    219300
2026-10-09    351300
2026-10-10    324900
2026-10-11    225000
2026-10-12    305500
2026-10-13    314100
2026-10-14    375200
2026-10-15    288500
2026-10-16    389700
2026-10-17    342000
2026-10-18    184600
2026-10-19    250100
2026-10-20    205600
2026-10-21    292400
2026-10-22    263500
2026-10-23    289100
2026-10-24    289800
2026-10-25    312300
2026-10-26    272800
2026-10-27    200400
2026-10-28    391000
2026-10-29    213300
2026-10-30    302800
2026-10-31    397300
Name: 출금액, dtype: int64


## 7. 최종 결과표 저장

In [8]:
# 매출과 식자재비 data 전처리
df_sales = pivot_df.reset_index()
df_mart = daily_mart_cost.reset_index()

df_mart = df_mart.rename(columns={"출금액":"식자재비"})

# 일자 기준으로 매출과 식자재비 데이터 합치기
final_df = pd.merge(
    df_sales, df_mart,
    left_on = "주문일자",
    right_on="거래일자",
    how = "left",
)

# 거래일자 열 삭제, 매출-식자재비 열 추가
final_df = final_df.drop(columns=["거래일자"])
final_df["매출-식자재비"] = final_df["합계"] - final_df["식자재비"]

# 결과 확인
print(final_df)

# 결과 저장
final_df.to_csv("result.csv", index=False, encoding="utf-8-sig")

          주문일자       배민       쿠팡       합계    식자재비  매출-식자재비
0   2026-10-01   959000   562000  1521000  197100  1323900
1   2026-10-02  1348000   904000  2252000  380400  1871600
2   2026-10-03  1294000   779000  2073000  247600  1825400
3   2026-10-04  1127000   928000  2055000  289300  1765700
4   2026-10-05  1103000   543000  1646000  214400  1431600
5   2026-10-06  1028000   576000  1604000  209500  1394500
6   2026-10-07  1227000   549000  1776000  332800  1443200
7   2026-10-08   946000   439000  1385000  219300  1165700
8   2026-10-09  1200000   835000  2035000  351300  1683700
9   2026-10-10  1379000   719000  2098000  324900  1773100
10  2026-10-11  1519000   979000  2498000  225000  2273000
11  2026-10-12   838000   622000  1460000  305500  1154500
12  2026-10-13   971000   539000  1510000  314100  1195900
13  2026-10-14   981000   663000  1644000  375200  1268800
14  2026-10-15  1068000   546000  1614000  288500  1325500
15  2026-10-16  1420000   776000  2196000  389700  18063